# Modul 12: Peramalan Probabilistik (P10–P90) & Optimasi BESS Mikrogrid
**Mata Kuliah:** Analisis Deret Waktu dan Peramalan (TEE-412) — S1 Teknik Elektro FT UNIB  
**Sub-CPMK 13 (Minggu 14):** Membangun peramalan probabilistik berbasis *Quantile Regression* (*Pinball Loss*), mengevaluasi PICP & PINAW, serta mengintegrasikannya ke optimasi penjadwalan baterai BESS (*Peak Shaving* & arbitrase tarif ToU).

In [ ]:
# !pip install -q scikit-learn scipy numpy pandas matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor
from scipy.optimize import linprog

plt.rcParams['axes.grid'] = True

def pinball_loss(y_true, y_pred, q):
    diff = y_true - y_pred
    return np.mean(np.maximum(q * diff, (q - 1) * diff))

## 1. Peramalan Probabilistik Daya PLTS (Kuantil P10, P50, P90)

In [ ]:
rng = np.random.default_rng(14)
n_days = 30
hours = np.tile(np.arange(24), n_days)
irrad_clear = np.where((hours >= 6) & (hours <= 18), 500 * np.sin(np.pi * (hours - 6) / 12), 0.0)
# Ketidakpastian awan heteroskedastik (lebih besar saat siang)
pv_actual = np.clip(irrad_clear * rng.uniform(0.65, 1.05, len(hours)) + rng.normal(0, 12, len(hours)), 0, None)

X = np.column_stack([
    np.sin(2 * np.pi * hours / 24),
    np.cos(2 * np.pi * hours / 24),
    irrad_clear,
])
split = 24 * 25
X_tr, y_tr = X[:split], pv_actual[:split]
X_te, y_te = X[split:split+24], pv_actual[split:split+24]

preds = {}
for q in [0.10, 0.50, 0.90]:
    gbr = GradientBoostingRegressor(loss='quantile', alpha=q, n_estimators=120, max_depth=3, random_state=42)
    gbr.fit(X_tr, y_tr)
    preds[q] = np.clip(gbr.predict(X_te), 0, None)
    print(f"Pinball Loss (tau={q:.2f}): {pinball_loss(y_te, preds[q], q):.2f} kW")

picp = np.mean((y_te >= preds[0.10]) & (y_te <= preds[0.90])) * 100
pinaw = np.mean(preds[0.90] - preds[0.10]) / (y_te.max() - y_te.min() + 1e-6) * 100
print(f"PICP (Target ~80%): {picp:.1f}% | PINAW: {pinaw:.1f}%")

## 2. Optimasi Penjadwalan BESS 24-Jam (Linear Programming / MPC Tarif ToU)

In [ ]:
# Parameter Mikrogrid & BESS
T = 24
h = np.arange(T)
load_fc = 280 + 90 * np.sin(2 * np.pi * (h - 6) / 24) + 110 * np.exp(-((h - 19) ** 2) / 6)
pv_p10 = preds[0.10]  # Gunakan batas konservatif P10 untuk keandalan!

# Tarif ToU PLN (Rp/kWh): WBP (17:00-22:00) mahal Rp 2.200, LWBP Rp 1.100
tariff = np.where((h >= 17) & (h <= 22), 2200.0, 1100.0)

E_max, E_min, E_init = 500.0, 100.0, 250.0  # kWh (SoC 20%-100%, awal 50%)
P_bat_max = 120.0  # kW
eta_c, eta_d = 0.95, 0.95

# Variabel keputusan: [P_grid(0..23), P_ch(0..23), P_dis(0..23), E(0..23)] -> total 4*T
c = np.concatenate([tariff, np.full(T, 10.0), np.full(T, 10.0), np.zeros(T)])

A_eq, b_eq = [], []
for t in range(T):
    # 1. Neraca daya: P_grid - P_ch + P_dis = load_fc - pv_p10
    row_bal = np.zeros(4 * T)
    row_bal[t] = 1.0
    row_bal[T + t] = -1.0
    row_bal[2 * T + t] = 1.0
    A_eq.append(row_bal)
    b_eq.append(load_fc[t] - pv_p10[t])

    # 2. Dinamika energi: E(t) - E(t-1) - eta_c*P_ch(t) + (1/eta_d)*P_dis(t) = 0
    row_e = np.zeros(4 * T)
    row_e[3 * T + t] = 1.0
    if t > 0:
        row_e[3 * T + t - 1] = -1.0
    row_e[T + t] = -eta_c
    row_e[2 * T + t] = 1.0 / eta_d
    A_eq.append(row_e)
    b_eq.append(E_init if t == 0 else 0.0)

bounds = (
    [(0, None)] * T
    + [(0, P_bat_max)] * T
    + [(0, P_bat_max)] * T
    + [(E_min, E_max * 0.90)] * T
)

sol = linprog(c, A_eq=np.array(A_eq), b_eq=np.array(b_eq), bounds=bounds, method='highs')
P_grid_opt = sol.x[:T]
P_ch_opt = sol.x[T : 2 * T]
P_dis_opt = sol.x[2 * T : 3 * T]
E_opt = sol.x[3 * T :]

cost_no_bess = np.sum(tariff * np.maximum(0, load_fc - pv_p10))
cost_with_bess = np.sum(tariff * P_grid_opt)
saving = cost_no_bess - cost_with_bess
print(f"Biaya Tanpa BESS : Rp {cost_no_bess:,.0f}")
print(f"Biaya Dengan BESS: Rp {cost_with_bess:,.0f} (Hemat Rp {saving:,.0f} / {saving/cost_no_bess*100:.1f}%)")

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
axes[0].fill_between(h, preds[0.10], preds[0.90], color='orange', alpha=0.25, label='Pita PV P10-P90')
axes[0].plot(h, y_te, 'k-o', ms=3, label='PV Aktual')
axes[0].plot(h, load_fc, 'b-', lw=2, label='Ramalan Beban')
axes[0].plot(h, P_grid_opt, 'g--', lw=2, label='Daya Beli PLN (Optimal)')
axes[0].set_ylabel('Daya (kW)')
axes[0].set_title('Optimasi Dispatch BESS Berbasis Ramalan Probabilistik P10')
axes[0].legend()

axes[1].bar(h, P_dis_opt - P_ch_opt, color=np.where(P_dis_opt >= P_ch_opt, '#10b981', '#3b82f6'), alpha=0.7, label='Net Discharge (+) / Charge (-)')
ax2 = axes[1].twinx()
ax2.plot(h, E_opt / E_max * 100, 'r-o', ms=4, label='SoC Baterai (%)')
axes[1].set_xlabel('Jam ke-t')
axes[1].set_ylabel('Daya BESS (kW)')
ax2.set_ylabel('SoC (%)')
axes[1].legend(loc='upper left')
ax2.legend(loc='upper right')
plt.tight_layout()
plt.show()

## 3. Tugas & Pertanyaan Analisis (C5–C6)
1. **[C5]** Bandingkan penghematan biaya jika optimasi menggunakan ramalan **P50 (median)** vs. **P10 (konservatif)**, lalu hitung risiko *shortfall* daya saat realisasi PV turun di bawah P50!
2. **[C6]** Tambahkan kendala terminal $E(23) \ge E_{\text{init}}$ agar baterai siap untuk siklus operasi hari berikutnya.